<a href="https://colab.research.google.com/github/yanchenliu-cxk/ESM/blob/main/horizyn_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 强制回到 Colab 根目录，防止多次运行导致的路径嵌套
%cd /content

# （可选清理）为了防止之前克隆的代码损坏，先删除旧文件夹，重新拉取干净的代码
!rm -rf horizyn
!git clone https://github.com/dayhofflabs/horizyn.git

# 2. 绝对路径进入项目根目录
%cd /content/horizyn

# 3. 安装包管理器并同步依赖（这一步会在内部自动生成并配置.venv）
!pip install uv
!uv sync

# 4. 使用 uv run 执行下载数据脚本（uv run 会自动寻找并使用正确的虚拟环境）
!uv run python scripts/download_data.py

# Diagnose missing file: Check contents of the data/sota directory
!ls -l data/sota

# 5. 启动模型训练
!uv run python train.py --config configs/sota.yaml

/content
Cloning into 'horizyn'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 102 (delta 12), reused 18 (delta 4), pack-reused 19 (from 1)
Receiving objects: 100% (102/102), 369.74 KiB | 19.46 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/horizyn
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 96 packages in 1ms
Prepared 1 package in 416ms
Installed 88 packages in 222ms
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.2
 + aiosignal==1.4.0
 + attrs==25.4.0
 + black==25.9.0
 + blosc2==3.11.0
 + certifi==2025.10.5
 + cfgv==3.4.0
 + charset-normalizer==3.4.4
 + click==8.3.0
 + coverage==7.11.0
 + distlib==0.4.0
 + drfp==0.3.7
 + et-xmlfile==2.0.0
 + filelock==3.20.0
 + flake8==7.3.0
 + frozenlist==1.8.0
 + fsspec==2025.10.0
 + greenlet==3.2.4
 + h5py==3.15.1
 + horizyn==1.0.0 (from file:///content/horizyn)
 + identify==2.

In [1]:
# 启动训练
!uv run python train.py --config configs/sota.yaml

/usr/bin/python3: can't open file '/content/train.py': [Errno 2] No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import yaml
import os

# 1. 定义配置文件路径和你的 Google Drive 目标路径
config_path = 'configs/sota.yaml'
drive_checkpoint_path = '/content/drive/MyDrive/Horizyn_Checkpoints'

# 2. 确保 Google Drive 中的目标文件夹存在（如果不存在则自动创建）
os.makedirs(drive_checkpoint_path, exist_ok=True)

# 3. 读取当前的 yaml 配置文件
with open(config_path, 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

# 4. 强制修改 logging 模块下的 checkpoint_dir 路径
if 'logging' in config:
    config['logging']['checkpoint_dir'] = drive_checkpoint_path
else:
    # 兼容性处理：如果原文件碰巧没有 logging 项，则主动创建
    config['logging'] = {'checkpoint_dir': drive_checkpoint_path}

# 5. 将修改后的配置重新写回文件，保持格式整洁
with open(config_path, 'w', encoding='utf-8') as file:
    yaml.dump(config, file, default_flow_style=False, sort_keys=False)

print(f"✅ 检查点保存路径已成功强制修改为: {drive_checkpoint_path}")

# 6. 使用 Linux 命令打印该行，验证是否写入硬盘
!echo "当前文件中的路径配置为："
!cat configs/sota.yaml | grep "checkpoint_dir"

✅ 检查点保存路径已成功强制修改为: /content/drive/MyDrive/Horizyn_Checkpoints
当前文件中的路径配置为：
  checkpoint_dir: /content/drive/MyDrive/Horizyn_Checkpoints


In [ ]:
import yaml
import os
import glob
import re

config_path = 'configs/sota.yaml'
drive_checkpoint_path = '/content/drive/MyDrive/Horizyn_Checkpoints'

# 1. 创建云盘保存路径（防断连的终极保障）
os.makedirs(drive_checkpoint_path, exist_ok=True)

# 2. 彻底且正确地覆写 YAML 核心配置
with open(config_path, 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

# 修复拼写错误，强制注入性能加速参数
if 'data' not in config: config['data'] = {}
config['data']['train_batch_size'] = 131072
if 'in_memory' in config['data']:
    del config['data']['in_memory'] # 自动清理错误拼写
config['data']['pin_memory'] = True

# 强制改为：每一轮结束都必须保存一次！
if 'logging' not in config: config['logging'] = {}
config['logging']['checkpoint_dir'] = drive_checkpoint_path
config['logging']['save_every_n_epochs'] = 1

with open(config_path, 'w', encoding='utf-8') as file:
    yaml.dump(config, file, default_flow_style=False, sort_keys=False)

print("✅ 配置文件已强制刷新：开启极大Batch、锁页内存、每 1 轮自动备份。")

# 3. 高级智能续训补丁：底层拦截并修改 Trainer.fit
ckpt_list = glob.glob(os.path.join(drive_checkpoint_path, '*.ckpt'))
latest_ckpt = max(ckpt_list, key=os.path.getctime) if ckpt_list else None

with open('train.py', 'r', encoding='utf-8') as f:
    content = f.read()

# 清理旧的拦截补丁（防止多次运行重复注入）
content = re.sub(r'# --- AUTO INJECTED RESUME PATCH ---.*?# ----------------------------------\n', '', content, flags=re.DOTALL)

if latest_ckpt:
    print(f"✅ 发现最新历史检查点：{os.path.basename(latest_ckpt)}")
    print("🚀 本次运行将无缝继承优化器与学习率，从断点处直接继续！")

    # 猴子补丁：直接在代码执行内存中强行挂载 ckpt_path，绝对不会引发语法错误
    patch_code = f"""# --- AUTO INJECTED RESUME PATCH ---
import pytorch_lightning as pl
_original_fit = pl.Trainer.fit
def _patched_fit(self, *args, **kwargs):
    kwargs['ckpt_path'] = r"{latest_ckpt}"
    print(f"\\n 成功挂载历史权重，从检查点恢复训练...\\n")
    return _original_fit(self, *args, **kwargs)
pl.Trainer.fit = _patched_fit
# ----------------------------------\n"""
    content = patch_code + content
else:
    print("⚠️ 云盘中暂无检查点。本次将从 Epoch 0 全新开始，跑完第 1 轮后就会自动产生备份！")

with open('train.py', 'w', encoding='utf-8') as f:
    f.write(content)

✅ 配置文件已强制刷新：开启极大Batch、锁页内存、每 1 轮自动备份。
⚠️ 云盘中暂无检查点。本次将从 Epoch 0 全新开始，跑完第 1 轮后就会自动产生备份！


In [ ]:
import yaml

config_path = 'configs/sota.yaml'

# 读取当前的配置文件
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

# 强制修改参数（确保是在 data 模块下）
config['data']['train_batch_size'] = 131072
config['data']['pin_memory'] = True

# 将修改后的配置重新写回文件
with open(config_path, 'w') as file:
    yaml.dump(config, file)

print("配置已成功更新并强制写入硬盘！")

# 打印出来验证一下
!cat configs/sota.yaml | grep -E "train_batch_size|pin_memory"

配置已成功更新并强制写入硬盘！
  pin_memory: true
  train_batch_size: 131072


In [ ]:
import IPython
from google.colab import output

display(IPython.display.Javascript('''
function ClickConnect(){
    // 自动点击连接按钮
    let btn = document.querySelector("colab-connect-button");
    if (btn!= null){
        console.log("正在保持 Colab 活跃...");
        btn.click();
    }
    // 自动处理可能弹出的确认对话框
    let ok_btn = document.getElementById('ok');
    if (ok_btn!= null){
        console.log("处理弹窗...");
        ok_btn.click();
    }
}
// 每 60 秒自动执行一次
setInterval(ClickConnect, 60000);
'''))
print("防断连脚本已在后台启动！")

<IPython.core.display.Javascript object>

防断连脚本已在后台启动！


In [ ]:
# 1. 确保在项目根目录
%cd /content/horizyn

# 2. 强制创建 sota.yaml 期望的文件夹路径
!mkdir -p data/sota

# 3. 如果文件错位下载到了外面，将它们移动到正确的文件夹中
!mv prots_t5.h5 data/sota/ 2>/dev/null || true
!mv *.csv data/sota/ 2>/dev/null || true
!mv data/*.csv data/sota/ 2>/dev/null || true
!mv data/*.h5 data/sota/ 2>/dev/null || true

# 4. 安全起见，再次执行官方下载脚本（如果文件已存在且完整，脚本通常会跳过下载）
!uv run python scripts/download_data.py

# 5. 再次启动模型训练
!uv run python train.py --config configs/sota.yaml

SyntaxError: invalid syntax (2978541721.py, line 10)

In [2]:
# 1. 强制回到 Colab 根目录，重新拉取干净的官方代码
%cd /content
!rm -rf horizyn
!git clone https://github.com/dayhofflabs/horizyn.git

# 2. 进入项目目录并利用 uv 极速同步环境依赖
%cd /content/horizyn
!pip install uv
!uv sync

# 3. 强制创建 sota 配置期望的数据结构，并重新下载 1GB 官方预训练数据集
!mkdir -p data/sota
!uv run python scripts/download_data.py

/content
Cloning into 'horizyn'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 102 (delta 12), reused 18 (delta 4), pack-reused 19 (from 1)
Receiving objects: 100% (102/102), 369.74 KiB | 19.46 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/horizyn
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 92.8 MB/s eta 0:00:00
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 96 packages in 0.91ms
Prepared 88 packages in 1m 16s
Installed 88 packages in 211ms
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.2
 + aiosignal==1.4.0
 + attrs==25.4.0
 + black==25.9.0
 + blosc2==3.11.0
 + certifi==2025.10.5
 + cfgv==3.4.0
 + charset-normalizer==3.4.4
 + click==8.3.0
 + coverage==7.11.0
 + distlib==0.4.0
 + drfp==0.3.7
 + et-xmlfile==2.0.0
 + filelock==3.20.0
 + flake8==7.3.0
 + frozenlist==1.8.0
 + fsspec==2025.10.0
 + greenlet==3

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import yaml
import os
import glob
import re

config_path = 'configs/sota.yaml'
drive_checkpoint_path = '/content/drive/MyDrive/Horizyn_Checkpoints'

# 1. 创建云盘保存路径（防断连的终极保障）
os.makedirs(drive_checkpoint_path, exist_ok=True)

# 2. 彻底且正确地覆写 YAML 核心配置
with open(config_path, 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

# 修复拼写错误，强制注入性能加速参数
if 'data' not in config: config['data'] = {}
config['data']['train_batch_size'] = 131072
if 'in_memory' in config['data']:
    del config['data']['in_memory'] # 自动清理之前不小心打错的拼写
config['data']['pin_memory'] = True

# 强制修改配置：每一轮结束都必须向云盘备份一次！
if 'logging' not in config: config['logging'] = {}
config['logging']['checkpoint_dir'] = drive_checkpoint_path
config['logging']['save_every_n_epochs'] = 1

with open(config_path, 'w', encoding='utf-8') as file:
    yaml.dump(config, file, default_flow_style=False, sort_keys=False)

print("✅ 配置文件已强制刷新：开启极大Batch、锁页内存、每 1 轮自动备份。")

# 3. 高级智能续训补丁：动态扫描云盘并拦截 Trainer.fit 注入断点路径
ckpt_list = glob.glob(os.path.join(drive_checkpoint_path, '*.ckpt'))
latest_ckpt = max(ckpt_list, key=os.path.getctime) if ckpt_list else None

with open('train.py', 'r', encoding='utf-8') as f:
    content = f.read()

# 清理旧的拦截补丁（防止多次运行重复注入）
content = re.sub(r'# --- AUTO INJECTED RESUME PATCH ---.*?# ----------------------------------\n', '', content, flags=re.DOTALL)

if latest_ckpt:
    print(f"✅ 发现最新历史检查点：{os.path.basename(latest_ckpt)}")
    print("🚀 本次运行将无缝继承优化器与学习率，从断点处直接继续！")

    # 猴子补丁（Monkey Patch）：在内存中重写 fit 方法，强制挂载 ckpt_path 恢复断点
    patch_code = f"""# --- AUTO INJECTED RESUME PATCH ---
import pytorch_lightning as pl
_original_fit = pl.Trainer.fit
def _patched_fit(self, *args, **kwargs):
    kwargs['ckpt_path'] = r"{latest_ckpt}"
    print(f"\\n 成功挂载历史权重，从检查点恢复训练...\\n")
    return _original_fit(self, *args, **kwargs)
pl.Trainer.fit = _patched_fit
# ----------------------------------\n"""
    content = patch_code + content
else:
    print("⚠️ 云盘中暂无检查点。本次将从 Epoch 0 全新开始。")

with open('train.py', 'w', encoding='utf-8') as f:
    f.write(content)

✅ 配置文件已强制刷新：开启极大Batch、锁页内存、每 1 轮自动备份。
✅ 发现最新历史检查点：last.ckpt
🚀 本次运行将无缝继承优化器与学习率，从断点处直接继续！


In [6]:
!uv run python train.py --config configs/sota.yaml

Loading config from: configs/sota.yaml

HORIZYN TRAINING CONFIGURATION
Seed: 42
Max Epochs: 100
Train Batch Size: 131072
Retrieval Batch Size: 128
Learning Rate: 0.0001
Weight Decay: 0.01
Model: DualContrastiveModel
Query Encoder: [2048, 4096, 4096, 512]
Target Encoder: [1024, 4096, 4096, 512]
Embedding Dim: 512
Loss: FullBatchMLNCELoss (beta=10.0)
Log Dir: logs
Checkpoint Dir: /content/drive/MyDrive/Horizyn_Checkpoints

Seed set to 42
Set random seed to: 42

Initializing data module...
Data module initialized.

Initializing model...
Total parameters: 50,349,057
Trainable parameters: 50,349,056

Setting up Lightning Trainer...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
Trainer configured for 100 epochs

STARTING TRAINING

You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more 

In [9]:
%%writefile resume_train.py
import os
import glob
import yaml
import torch
import pytorch_lightning as pl

# 1. 精准定位云盘中的最新断点权重
drive_checkpoint_path = '/content/drive/MyDrive/Horizyn_Checkpoints'
ckpt_list = glob.glob(os.path.join(drive_checkpoint_path, '*.ckpt'))
latest_checkpoint = max(ckpt_list, key=os.path.getctime) if ckpt_list else None

# 2. 核心加载与启动说明
print("="*50)
if latest_checkpoint:
    print(f"🚀 [智能断点续训] 成功锁定云盘最新历史进度: {os.path.basename(latest_checkpoint)}")
    print("🎯 系统将完美继承优化器状态、学习率和参数，直接恢复训练！")
else:
    print("⚠️ 未发现历史检查点，将从 Epoch 0 开始全新的全量训练。")
print("="*50)

# 3. 利用内存加载原项目模块进行标准训练
from horizyn.data_module import ReactionProteinDataModule
from horizyn.lightning_module import DualContrastiveModel

# 读取并强制应用优化后的参数配置 (Batch Size: 131072, pin_memory: True)
with open('configs/sota.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['data']['train_batch_size'] = 131072
config['data']['pin_memory'] = True
config['logging']['checkpoint_dir'] = drive_checkpoint_path
config['logging']['save_every_n_epochs'] = 1

# 实例化官方的数据模块与核心对齐模型
data_module = ReactionProteinDataModule(config['data'])
model = DualContrastiveModel(config['model'])

# 初始化核心 Trainer，绑定每一轮自动备份的回调机制
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath=drive_checkpoint_path,
    filename="horizyn-epoch={epoch:02d}",
    every_n_epochs=1,
    save_top_k=3,
    save_last=True
)

trainer = pl.Trainer(
    max_epochs=100,
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_callback],
    default_root_dir=drive_checkpoint_path
)

# 4. 一键编译加速
model = torch.compile(model)

if latest_checkpoint:
    # 显式传入 ckpt_path 参数，强制要求 Lightning 从断点恢复续训
    trainer.fit(model, datamodule=data_module, ckpt_path=latest_checkpoint)
else:
    trainer.fit(model, datamodule=data_module)

Overwriting resume_train.py


In [10]:
# 强制在包含所有依赖的虚拟环境中运行刚刚保存的脚本
!uv run python resume_train.py

🚀 [智能断点续训] 成功锁定云盘最新历史进度: last.ckpt
🎯 系统将完美继承优化器状态、学习率和参数，直接恢复训练！
Traceback (most recent call last):
  File "/content/horizyn/resume_train.py", line 22, in <module>
    from horizyn.data_module import ReactionProteinDataModule
ImportError: cannot import name 'ReactionProteinDataModule' from 'horizyn.data_module' (/content/horizyn/horizyn/data_module.py)


In [11]:
import yaml
import os
import glob
import re

config_path = 'configs/sota.yaml'
drive_checkpoint_path = '/content/drive/MyDrive/Horizyn_Checkpoints'

# 1. 确保云盘目录完好
os.makedirs(drive_checkpoint_path, exist_ok=True)

# 2. 强力刷新 YAML 配置文件，确保参数 100% 正确写入硬盘
with open(config_path, 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

if 'data' not in config: config['data'] = {}
config['data']['train_batch_size'] = 131072
config['data']['pin_memory'] = True
if 'in_memory' in config['data']:
    del config['data']['in_memory']  # 彻底自动抠除之前拼写错误的键

# 确保每一轮都自动向云盘写入备份
if 'logging' not in config: config['logging'] = {}
config['logging']['checkpoint_dir'] = drive_checkpoint_path
config['logging']['save_every_n_epochs'] = 1

with open(config_path, 'w', encoding='utf-8') as file:
    yaml.dump(config, file, default_flow_style=False, sort_keys=False)

print("✅ 1. configs/sota.yaml 核心配置已强制刷新并固化。")

# 3. 动态扫描云盘，锁定您刚刚断开前保存的最新第 69 轮或 last.ckpt 权重文件
ckpt_list = glob.glob(os.path.join(drive_checkpoint_path, '*.ckpt'))
latest_ckpt = max(ckpt_list, key=os.path.getctime) if ckpt_list else None

# 4. 读取官方的 train.py 并注入全自动拦截补丁
with open('train.py', 'r', encoding='utf-8') as f:
    train_code = f.read()

# 清理可能残存的旧历史补丁代码，防止代码重复叠加
train_code = re.sub(r'# --- AUTO INJECTED RESUME PATCH ---.*?# ----------------------------------\n', '', train_code, flags=re.DOTALL)

if latest_ckpt:
    print(f"✅ 2. 成功捕获云盘最新历史断点：{os.path.basename(latest_ckpt)}")
    print("🚀 正在向官方原始 train.py 注入无缝续训拦截器...")

    # 针对官方底层实际使用的 'lightning.pytorch' 模块进行精准运行时动态拦截
    patch_code = f"""# --- AUTO INJECTED RESUME PATCH ---
import lightning.pytorch as pl
_orig_fit = pl.Trainer.fit
def _patched_fit(self, model, *args, **kwargs):
    kwargs['ckpt_path'] = r"{latest_ckpt}"
    print("\\n🚀🚀🚀 [智能断点续训] 成功拦截并挂载历史权重，正在恢复第 69 轮之后的计算...\\n")
    return _orig_fit(self, model, *args, **kwargs)
pl.Trainer.fit = _patched_fit
# ----------------------------------\n"""
    train_code = patch_code + train_code
else:
    print("⚠️ 2. 未在云盘对应路径下发现检查点，本次将从第 0 轮全新启动。")

with open('train.py', 'w', encoding='utf-8') as f:
    f.write(train_code)

print("\n配置全部就绪！请直接运行下方启动单元格。")

✅ 1. configs/sota.yaml 核心配置已强制刷新并固化。
✅ 2. 成功捕获云盘最新历史断点：last.ckpt
🚀 正在向官方原始 train.py 注入无缝续训拦截器...

配置全部就绪！请直接运行下方启动单元格。


In [12]:
# 直接在已经包含所有依赖的隔离虚拟环境中启动官方训练程序
!uv run python train.py --config configs/sota.yaml

Loading config from: configs/sota.yaml

HORIZYN TRAINING CONFIGURATION
Seed: 42
Max Epochs: 100
Train Batch Size: 131072
Retrieval Batch Size: 128
Learning Rate: 0.0001
Weight Decay: 0.01
Model: DualContrastiveModel
Query Encoder: [2048, 4096, 4096, 512]
Target Encoder: [1024, 4096, 4096, 512]
Embedding Dim: 512
Loss: FullBatchMLNCELoss (beta=10.0)
Log Dir: logs
Checkpoint Dir: /content/drive/MyDrive/Horizyn_Checkpoints

Seed set to 42
Set random seed to: 42

Initializing data module...
Data module initialized.

Initializing model...
Total parameters: 50,349,057
Trainable parameters: 50,349,056

Setting up Lightning Trainer...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
Trainer configured for 100 epochs

STARTING TRAINING


🚀🚀🚀 [智能断点续训] 成功拦截并挂载历史权重，正在恢复第 69 轮之后的计算...

You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will 